# PUMP analog features

Adapted from `clean_extract_analog_features(3).ipynb`. The FAN numerical feature definitions are retained where applicable; PUMP loading, identities, states, and region selection replace the FAN-specific parts.

Run the cells in order. Put the two supplied PUMP CSVs beside the notebook, or edit `META_PATH` and `TIMESTAMPS_PATH`. The raw sensor files must exist at the paths in the metadata; use `PATH_REPLACEMENTS` if the drive/root changed. Dependencies: `numpy pandas polars scipy matplotlib tqdm`.

- Default: healthy + single faults, determined from nonzero fault-code positions. The supplied data selects **315 paired samples / 18 classes**; `INCLUDE_MULTIFAULTS=True` selects all **613 pairs / 38 classes**.
- Use every interval between consecutive signalwise changepoints: **Air has regions 0–1; Water has regions 0–3**. Intervals are `[start, end)`. Supplied indices refer to raw-file rows; rows are never filtered or reordered before slicing.
- One output row per `fault_id@pump_id@sample_id`, with both states joined by ID. Example feature: `CV1_air_0_col_0_rms`. Region 0 means the first interval; it does not label an operating condition by itself.
- Required channels match the supplied PUMP reference: AC1/GYR have three axes; VRY has two columns; CV1, CTB, CTR, SMG, PRS and WTF use column 0. PRS/WTF apply only to Water. No extra channels are silently added.
- The uploaded metadata references **legacy data only**. Legacy `Timestamp` is seconds. A future merged file may use `firmware_timestamp` or `F_Timestamp` in microseconds. New electrical channel equivalents must be explicitly set in `NEW_RAW_COLUMNS`; the older inference mapping is not assumed to prove physical/calibration equivalence.
- Output files are saved in `pump_features`. All four notebooks are independent and use the same sample keys; no inference config, checkpoint or helper file is needed.

The Hann window, 30–70 Hz FFT band, dominant peak, peak width and spectral statistics come from FAN. The window is **2^16 samples instead of FAN's 2^18**, centred inside each region with 0.25 seconds excluded at each edge; several supplied PUMP regions cannot fit the original 2^18 window. Pressure and flow are covered by the statistics notebook, not mains-frequency FFT features. Missing peaks/constant analog channels raise a contextual error.

Optional plot after loading: `plot_fft(index=0, sig="CV1", col="0", region=0);`

In [1]:
import os

# Conservative thread limits to reduce sustained CPU load.
# Restart the kernel before running this notebook.
os.environ["POLARS_MAX_THREADS"] = "1"
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"

import gc
import math
import pandas as pd
import numpy as np
import polars as pl
import matplotlib.pyplot as plt


from tqdm import tqdm
from scipy.signal import find_peaks, windows, peak_widths
from scipy.fft import rfft, rfftfreq
import scipy.stats as sci_st

In [2]:
from pathlib import Path

# Only edit these paths if the CSVs are not beside this notebook.
META_PATH = Path("pump_meta_combined_all_faults_mapped.csv")
TIMESTAMPS_PATH = Path("pump_signalwise_timestamp_combined.csv")
OUTPUT_DIR = Path("pump_features")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
INCLUDE_MULTIFAULTS = False  # Same healthy + single-fault selection as FAN.
STATES = ("air", "water")
REGION_EDGE_SECONDS = 0.25

# Optional relocation of the raw files; leave empty if the CSV paths are valid.
PATH_REPLACEMENTS = {}  # e.g. {r"D:\VGuard Ahemedebad Data": r"E:\PumpData"}

# These are the legacy columns used by the supplied PUMP feature/config files.
SIGNAL_COLUMNS = {
    "AC1": ["0", "1", "2"], "GYR": ["0", "1", "2"],
    "CTB": ["0"], "CTR": ["0"], "CV1": ["0"], "VRY": ["0", "1"],
    "SMG": ["0"], "PRS": ["0"], "WTF": ["0"],
}
SIGNALS_BY_STATE = {
    "air": ["SMG", "AC1", "VRY", "CV1", "GYR", "CTB", "CTR"],
    "water": ["SMG", "AC1", "VRY", "CV1", "GYR", "CTB", "CTR", "PRS", "WTF"],
}

# The supplied combined metadata contains legacy files only. For future raw
# analog.csv files, explicitly confirm electrical channel equivalents here.
# No old/new calibration or electrical equivalence is inferred from filenames.
NEW_RAW_COLUMNS = {
    "AC1": {"0": "ac1", "1": "ac2", "2": "ac3"},
    "GYR": {"0": "gy1", "1": "gy2", "2": "gy3"},
    "SMG": {"0": "mag"}, "PRS": {"0": "prs"}, "WTF": {"0": "wtf"},
    "CV1": {}, "VRY": {}, "CTB": {}, "CTR": {},
}

meta = pd.read_csv(META_PATH, dtype={"fault_id": str, "pump_id": str, "sample_id": str})
orig_ts = pd.read_csv(TIMESTAMPS_PATH, dtype=str)
meta["state"] = meta["state"].str.strip().str.lower()
orig_ts["state"] = orig_ts["state"].str.strip().str.lower()
for col in ["fault_id", "pump_id", "sample_id"]:
    meta[col] = meta[col].str.strip()
meta["sample_id_key"] = meta[["fault_id", "pump_id", "sample_id"]].agg("@".join, axis=1)
meta["pump_id_key"] = meta[["fault_id", "pump_id"]].agg("@".join, axis=1)
meta["state_id_key"] = meta["sample_id_key"] + "@" + meta["state"]
active_faults = meta["fault_id"].str.rstrip("_").map(lambda x: sum(c != "0" for c in x))
meta["fault_type"] = np.where(active_faults == 0, "nofault", np.where(active_faults == 1, "single", "multi"))
if not INCLUDE_MULTIFAULTS:
    meta = meta[meta["fault_type"] != "multi"].copy()
meta = meta.reset_index(drop=True)
if meta.empty or meta["state_id_key"].duplicated().any():
    raise ValueError("Metadata is empty or has duplicate sample/state keys.")
if not set(meta["state"]).issubset(STATES):
    raise ValueError("Unrecognized state in PUMP metadata.")
if orig_ts["id"].duplicated().any():
    raise ValueError("Duplicate signalwise changepoint IDs.")

orig_ts_lookup = {}
for row in orig_ts.itertuples(index=False):
    if not row.id.startswith(row.key + "_") or not row.id.endswith("@" + row.state):
        raise ValueError(f"Inconsistent signal/state in timestamp ID: {row.id}")
    cp = np.array(row.indices.split("#"), dtype=np.int64)
    if len(cp) < 2 or cp[0] < 0 or np.any(np.diff(cp) <= 0):
        raise ValueError(f"Invalid changepoints: {row.id}")
    orig_ts_lookup[row.id] = cp

# Pair Air/Water by explicit IDs, never by row position.
for sample_id, group in meta.groupby("sample_id_key", sort=False):
    if set(group["state"]) != set(STATES):
        raise ValueError(f"Missing Air/Water state: {sample_id}")
region_counts = {}
for index, row in meta.iterrows():
    for sig in SIGNALS_BY_STATE[row["state"]]:
        if sig not in meta or pd.isna(row[sig]) or not str(row[sig]).strip():
            raise ValueError(f"Missing {sig} path for {row['state_id_key']}")
        cp_key = f"{sig}_{row['state_id_key']}"
        if cp_key not in orig_ts_lookup:
            raise ValueError(f"Missing changepoints: {cp_key}")
        count = len(orig_ts_lookup[cp_key]) - 1
        previous = region_counts.setdefault(row["state"], count)
        if previous != count:
            raise ValueError(f"Inconsistent number of regions: {cp_key}")

print(f"{meta['sample_id_key'].nunique()} paired samples; {meta['fault_id'].nunique()} classes")
print("Regions per state:", region_counts)
meta.head()

315 paired samples; 18 classes
Regions per state: {'air': 2, 'water': 4}


,fault_id,pump_id,sample_id,state,AC1,GYR,CTB,CTR,CV1,VRY,PRS,SMG,WTF,sample_id_key,pump_id_key,state_id_key,fault_type
0,00000000000_,1,1,air,D:\VGuard Ahemedebad Data\Primary_20230727-31\...,D:\VGuard Ahemedebad Data\Primary_20230727-31\...,D:\VGuard Ahemedebad Data\Primary_20230727-31\...,D:\VGuard Ahemedebad Data\Primary_20230727-31\...,D:\VGuard Ahemedebad Data\Primary_20230727-31\...,D:\VGuard Ahemedebad Data\Primary_20230727-31\...,NaN,D:\VGuard Ahemedebad Data\Primary_20230727-31\...,NaN,00000000000_@1@1,00000000000_@1,00000000000_@1@1@air,nofault
1,00000000000_,1,1,water,D:\VGuard Ahemedebad Data\Primary_20230727-31\...,D:\VGuard Ahemedebad Data\Primary_20230727-31\...,D:\VGuard Ahemedebad Data\Primary_20230727-31\...,D:\VGuard Ahemedebad Data\Primary_20230727-31\...,D:\VGuard Ahemedebad Data\Primary_20230727-31\...,D:\VGuard Ahemedebad Data\Primary_20230727-31\...,D:\VGuard Ahemedebad Data\Primary_20230727-31\...,D:\VGuard Ahemedebad Data\Primary_20230727-31\...,D:\VGuard Ahemedebad Data\Primary_20230727-31\...,00000000000_@1@1,00000000000_@1,00000000000_@1@1@water,nofault
2,00000000000_,1,4,air,D:\VGuard Ahemedebad Data\Primary_20230726\No ...,D:\VGuard Ahemedebad Data\Primary_20230726\No ...,D:\VGuard Ahemedebad Data\Primary_20230726\No ...,D:\VGuard Ahemedebad Data\Primary_20230726\No ...,D:\VGuard Ahemedebad Data\Primary_20230726\No ...,D:\VGuard Ahemedebad Data\Primary_20230726\No ...,NaN,D:\VGuard Ahemedebad Data\Primary_20230726\No ...,NaN,00000000000_@1@4,00000000000_@1,00000000000_@1@4@air,nofault
3,00000000000_,1,4,water,D:\VGuard Ahemedebad Data\Primary_20230726\No ...,D:\VGuard Ahemedebad Data\Primary_20230726\No ...,D:\VGuard Ahemedebad Data\Primary_20230726\No ...,D:\VGuard Ahemedebad Data\Primary_20230726\No ...,D:\VGuard Ahemedebad Data\Primary_20230726\No ...,D:\VGuard Ahemedebad Data\Primary_20230726\No ...,D:\VGuard Ahemedebad Data\Primary_20230726\No ...,D:\VGuard Ahemedebad Data\Primary_20230726\No ...,D:\VGuard Ahemedebad Data\Primary_20230726\No ...,00000000000_@1@4,00000000000_@1,00000000000_@1@4@water,nofault
4,00000000000_,2,1,air,D:\VGuard Ahemedebad Data\Primary_20230726\No ...,D:\VGuard Ahemedebad Data\Primary_20230726\No ...,D:\VGuard Ahemedebad Data\Primary_20230726\No ...,D:\VGuard Ahemedebad Data\Primary_20230726\No ...,D:\VGuard Ahemedebad Data\Primary_20230726\No ...,D:\VGuard Ahemedebad Data\Primary_20230726\No ...,NaN,D:\VGuard Ahemedebad Data\Primary_20230726\No ...,NaN,00000000000_@2@1,00000000000_@2,00000000000_@2@1@air,nofault


In [3]:
def resolve_path(value):
    path = str(value)
    for old_root, new_root in PATH_REPLACEMENTS.items():
        if path.startswith(old_root):
            path = str(new_root) + path[len(old_root):]
            break
    if os.name != "nt":
        path = path.replace("\\", "/")
    return path


def read_pump_signal(index, sig):
    """Read one signal without removing/reordering rows referenced by changepoints."""
    path = resolve_path(meta.loc[index, sig])
    header = pd.read_csv(path, nrows=0).columns.tolist()
    new_time = next((c for c in ("firmware_timestamp", "F_Timestamp") if c in header), None)
    timecol = new_time or "Timestamp"
    if timecol not in header:
        raise ValueError(f"Timestamp column missing: {path}")
    mapping = {}
    for col in SIGNAL_COLUMNS[sig]:
        # Prefer canonical signal names in merged files. Bare column_0/0
        # names apply only to separate legacy signal files.
        candidates = [f"{sig}_column_{col}"]
        if not new_time:
            candidates += [f"column_{col}", col]
        else:
            configured = NEW_RAW_COLUMNS.get(sig, {}).get(col)
            if configured:
                candidates.append(configured)
        found = next((c for c in candidates if c in header), None)
        if found is None:
            raise ValueError(f"Missing/unmapped {sig} column {col}: {path}. "
                             "For new raw electrical files, set NEW_RAW_COLUMNS explicitly.")
        mapping[found] = col
    orig = pl.read_csv(path, columns=[timecol] + list(mapping), n_threads=1,
                       low_memory=True, rechunk=False).rename({timecol: "Timestamp", **mapping})
    if new_time:
        orig = orig.with_columns((pl.col("Timestamp") / 1e6).alias("Timestamp"))
    times = orig["Timestamp"].to_numpy()
    valid_positions = np.flatnonzero(np.isfinite(times) & (times >= 0))
    if len(valid_positions) < 2:
        raise ValueError(f"Not enough valid timestamps: {path}")
    first, last = valid_positions[[0, -1]]
    duration = times[last] - times[first]
    if duration <= 0:
        raise ValueError(f"Invalid timestamp duration: {path}")
    fs = (last - first) / duration  # Legacy Timestamp is seconds; firmware time is microseconds.
    cp_key = f"{sig}_{meta.loc[index, 'state_id_key']}"
    cp = orig_ts_lookup[cp_key]
    if cp[-1] > orig.height:
        raise ValueError(f"Changepoints exceed raw-file row count: {cp_key}")
    return orig, fs, cp


def region_window(cp, region, fs, dist):
    """Fixed-size central window, with a margin inside both changepoints."""
    start, end = map(int, cp[region:region + 2])
    edge = int(np.ceil(REGION_EDGE_SECONDS * fs))
    available = end - start - 2 * edge
    if available < dist:
        raise ValueError(f"Region {region}: needs {dist} samples plus edge margins; "
                         f"only {available} usable samples. No crossing into another region.")
    ll = start + edge + (available - dist) // 2
    return ll, ll + dist


def finite_window(values, context):
    x = np.asarray(values, dtype=np.float64)
    if len(x) == 0 or not np.all(np.isfinite(x)):
        raise ValueError(f"Empty/non-finite feature window: {context}")
    return x


def feature_prefix(index, sig, region, col):
    return f"{sig}_{meta.loc[index, 'state']}_{region}_col_{col}"


def collect_features(extractor, description):
    """One output row per fault/pump/sample; state and region live in column names."""
    rows = []
    for sample_id, group in tqdm(meta.groupby("sample_id_key", sort=False),
                                 total=meta["sample_id_key"].nunique(), desc=description):
        row = {}
        for index in group.index:
            package = extractor(index)
            overlap = set(row).intersection(package)
            if overlap:
                raise ValueError(f"Duplicate state/region features: {overlap}")
            row.update(package)
        identity = group.iloc[0]
        for key in ["sample_id_key", "fault_id", "pump_id", "sample_id", "pump_id_key", "fault_type"]:
            row[key] = identity[key]
        rows.append(row)
        gc.collect()
    result = pd.DataFrame(rows)
    if result.empty or result.isna().any().any():
        raise ValueError("Empty/incomplete feature table; inspect sample regions and signals.")
    numeric = result.select_dtypes(include=np.number)
    if not np.isfinite(numeric.to_numpy()).all():
        raise ValueError("Non-finite feature values; inspect the raw signals.")
    return result

In [4]:
def plot_fft(index=0, sig="CV1", col="0", region=0, dist=2**16,
             xlim=(30, 70), plot_fft_graph=True, figsize=(18, 4),
             apply_window="hann", orig=None, fs=None, cp=None):
    if orig is None:
        orig, fs, cp = read_pump_signal(index, sig)
    sample_id = meta.loc[index, "state_id_key"]
    orig_sample_freq = fs
    ll, ul = region_window(cp, region, fs, dist)
    x = finite_window(orig[col].slice(ll, ul - ll).to_numpy(), f"{sample_id}/{sig}/{region}/{col}")
    if np.ptp(x) <= 1e-12 * max(1.0, float(np.max(np.abs(x)))):
        raise ValueError(f"Constant analog signal: {sample_id}/{sig}/{region}/{col}")
    window = []
    conv = lambda index: x_freq[index].item()
    x = x - np.mean(x)
    x_old = x
    if apply_window != None:
        if apply_window == 'hann':
            window = windows.hann(x.shape[0])
            x = x * window
        elif apply_window == 'hamming':
            window = windows.hamming(x.shape[0])
            x = x * window
        elif apply_window == 'triangular':
            window = windows.triang(x.shape[0])
            x = x * window
        elif apply_window == 'parzen':
            window = windows.parzen(x.shape[0])
            x = x * window
        elif apply_window == 'taylor':
            window = windows.taylor(x.shape[0])
            x = x * window
        elif apply_window == 'cosine':
            window = windows.cosine(x.shape[0])
            x = x * window
        elif apply_window == 'nuttall':
            window = windows.nuttall(x.shape[0])
            x = x * window
    _fft = np.abs(rfft(x, dist * 2))
    _fft_freq = rfftfreq(dist * 2, d=1 / orig_sample_freq)
    mask = np.bitwise_and(_fft_freq < xlim[1], _fft_freq > xlim[0])
    y_amp = _fft
    raw_amp = _fft
    denominator = float(np.max(y_amp[4:]))
    if denominator <= 1e-12 or not np.any(mask):
        raise ValueError(f"No usable FFT energy/bins: {sample_id}/{sig}/{region}/{col}")
    y_amp = y_amp / denominator
    x_freq = _fft_freq[mask]
    y_amp = y_amp[mask]
    raw_amp = raw_amp[mask]
    mag_sum = np.sum(y_amp)
    if mag_sum > 1e-12:
        mag_norm = y_amp / mag_sum
    else:
        mag_norm = np.zeros_like(y_amp)
    spectral_centroid = float(np.sum(x_freq * mag_norm))
    spectral_bandwidth = float(np.sqrt(np.sum((x_freq - spectral_centroid) ** 2 * mag_norm)))
    spectral_entropy = float(sci_st.entropy(mag_norm + 1e-12))
    peak_indices, _ = find_peaks(y_amp, height=np.max(y_amp) * 0.1)
    if len(peak_indices) == 0:
        raise ValueError(f"No interior FFT peak in {xlim}: {sample_id}/{sig}/{region}/{col}")
    peak_freqs = x_freq[peak_indices]
    peak_mags = y_amp[peak_indices]
    peak_mags_raw = raw_amp[peak_indices]
    top_peaks_idx = np.argsort(peak_mags)[-5:][::-1]
    top_peak_freqs = [float(round(val, 2)) for val in peak_freqs[top_peaks_idx][:1]]
    top_peak_mags = [float(round(val, 2)) for val in peak_mags[top_peaks_idx][:1]]
    top_peak_mags_raw = [float(round(val, 2)) for val in peak_mags_raw[top_peaks_idx][:1]]
    top_idx = peak_indices[np.argmax(peak_mags)]
    peak_to_median = float(y_amp[top_idx] / (np.median(y_amp) + 1e-12))
    peak_energy_ratio = float(raw_amp[top_idx] ** 2 / (np.sum(raw_amp ** 2) + 1e-12))
    results_half = peak_widths(y_amp, peak_indices[np.where(peak_mags == max(peak_mags))], rel_height=0.85)
    x_lims = [conv(math.floor(results_half[-2][0].item())), conv(math.ceil(results_half[-1][0].item()))]
    if plot_fft_graph:
        fig, ax = plt.subplots(1, 2, figsize=figsize)
        x_old = x_old - x_old.mean()
        fig.suptitle(f'FFT {sig} column {col}, region {region}: {sample_id}    Samples: {dist}    Samp Freq: {orig_sample_freq}   Window: {apply_window}')
        ax[0].plot(x_old / max(x_old))
        if len(window) != 0:
            ax[0].plot(window, 'green')
            ax[0].plot(window * -1, 'green')
            ax[0].plot(x_old / max(x_old) * window, 'orange')
        ax[1].plot(x_freq, y_amp)
        ax[1].set_xlim(xlim)
        ax[1].set_ylim((0, 1.1))
        ax[1].set_xlabel('Freq')
        ax[1].set_ylabel('Amp')
        ax[1].scatter(top_peak_freqs, top_peak_mags, color='red')
        for i, txt in enumerate(top_peak_mags):
            ax[1].annotate(f'{txt}, Freq: {top_peak_freqs[i]}', (top_peak_freqs[i], top_peak_mags[i]), bbox=dict(boxstyle='round,pad=0.2', fc='white', ec='gray', alpha=0.8), fontsize=9)
        ax[1].vlines(top_peak_freqs, ymax=top_peak_mags, ymin=0, color='black', linestyles='--')
        ax[1].hlines(results_half[1], x_lims[0], x_lims[1], color='C2')
    return top_peak_freqs + top_peak_mags_raw + x_lims + [sci_st.gmean(y_amp), sci_st.hmean(y_amp), sci_st.pmean(y_amp, 2), sci_st.pmean(y_amp, 3), sci_st.kurtosis(y_amp), sci_st.skew(y_amp), peak_to_median, peak_energy_ratio, spectral_centroid, spectral_bandwidth, spectral_entropy]

In [5]:
ANALOG_SIGNALS = ["CV1", "CTB", "CTR", "VRY", "SMG"]
ANALOG_DIST = 2**16
FFT_FEATURES = ["peak_freq", "peak_magn_raw", "peak_width_start", "peak_width_end",
                "gmean", "hmean", "pmean_2", "pmean_3", "fft_kurt", "fft_skew",
                "peak_to_median", "peak_energy_ratio", "spectral_centroid",
                "spectral_bandwidth", "spectral_entropy"]

def extract_analog_features(index):
    feats = {}
    for sig in ANALOG_SIGNALS:
        orig, fs, cp = read_pump_signal(index, sig)
        for region in range(len(cp) - 1):
            for col in SIGNAL_COLUMNS[sig]:
                values = plot_fft(index, sig, col, region, dist=ANALOG_DIST,
                                  plot_fft_graph=False, orig=orig, fs=fs, cp=cp)
                key = feature_prefix(index, sig, region, col)
                feats.update({f"{key}_{name}": value for name, value in zip(FFT_FEATURES, values)})
                feats[f"{key}_peak_width"] = values[3] - values[2]
        del orig
    return feats

In [6]:
df_ctvt = collect_features(extract_analog_features, "PUMP analog")
df_ctvt.head()

PUMP analog: 100%|██████████| 315/315 [19:36<00:00,  3.74s/it]


,CV1_air_0_col_0_peak_freq,CV1_air_0_col_0_peak_magn_raw,CV1_air_0_col_0_peak_width_start,CV1_air_0_col_0_peak_width_end,CV1_air_0_col_0_gmean,CV1_air_0_col_0_hmean,CV1_air_0_col_0_pmean_2,CV1_air_0_col_0_pmean_3,CV1_air_0_col_0_fft_kurt,CV1_air_0_col_0_fft_skew,...,SMG_water_3_col_0_spectral_centroid,SMG_water_3_col_0_spectral_bandwidth,SMG_water_3_col_0_spectral_entropy,SMG_water_3_col_0_peak_width,sample_id_key,fault_id,pump_id,sample_id,pump_id_key,fault_type
0,50.26,7665381.60,49.440567,50.882584,0.001060,0.000623,0.127040,0.238792,41.761494,6.422820,...,49.948904,2.081803,2.109061,2.444471,00000000000_@1@1,00000000000_,1,1,00000000000_@1,nofault
1,49.52,7714524.96,48.903735,50.330094,0.001119,0.000468,0.124332,0.234166,42.320159,6.464521,...,49.961638,2.203412,2.125355,2.447753,00000000000_@1@4,00000000000_,1,4,00000000000_@1,nofault
2,49.46,7864912.51,48.853798,50.278701,0.000672,0.000353,0.126079,0.237518,42.343198,6.467298,...,50.006336,2.109620,2.130929,2.447285,00000000000_@2@1,00000000000_,2,1,00000000000_@2,nofault
3,49.41,8213901.66,48.600614,50.227413,0.000869,0.000469,0.123520,0.232889,42.596228,6.485010,...,50.061475,1.944778,2.109164,2.447285,00000000000_@2@2,00000000000_,2,2,00000000000_@2,nofault
4,49.93,7704699.84,49.311606,50.749861,0.000767,0.000394,0.125754,0.236529,41.898892,6.434697,...,49.902002,2.117165,2.129940,2.447512,00000000000_@2@3,00000000000_,2,3,00000000000_@2,nofault


In [7]:
df_ctvt.to_csv(OUTPUT_DIR / "PUMP_ctvt_singlefault_peaks_freq.csv", index=False)
print("Saved", df_ctvt.shape)

Saved (315, 582)
